# Factory Pattern

### THE INTERFACE (Abstract Product)

In [8]:
from abc import ABC, abstractmethod

class Transport(ABC):
    @abstractmethod
    def deliver(self) -> str:
        """Abstract method ensuring all products behave the same way."""
        pass

### THE CONCRETE PRODUCTS

In [5]:
class Truck(Transport):
    def deliver(self) -> str:
        return "Delivering cargo by land in a box."

class Ship(Transport):
    def deliver(self) -> str:
        return "Delivering cargo by sea in a container."

class Drone(Transport):
    def deliver(self) -> str:
        return "Delivering small parcel by air."

### THE FACTORY

In [18]:
class LogisticsFactory:

    __TRANSPORTS = {
        "road": Truck(),
        "sea": Ship(),
        "air": Drone()
    }
    
    @staticmethod
    def get_transport(transport_type: str) -> Transport:
        """
        Factory Method.
        
        param transport_type: "road", "sea", or "air"
        returns: A Concrete Transport Object
        """
        # Using the modern 'match' statement (Python 3.10+)
        match transport_type.lower():
            case "road":
                return Truck()
            case "sea":
                return Ship()
            case "air":
                return Drone()
            case _:
                raise ValueError(f"Unknown transport type: {transport_type}")
                
    @staticmethod
    def get_transport_v2(transport_type: str) -> Transport:
        """
        Factory Method.
        
        param transport_type: "road", "sea", or "air"
        returns: A Concrete Transport Object
        """
        transport = LogisticsFactory.__TRANSPORTS.get(transport_type)
        if not transport:
            raise ValueError(f"Unknown transport type: {transport_type}")
        return transport
        

### CLIENT CODE

In [19]:
def main():
    factory = LogisticsFactory()

    # Define a list of requests to process
    requests: list[str] = ["road", "sea", "air", "space"]

    print("--- Logistics App Started ---\n")

    for req in requests:
        try:
            # The client doesn't know (or care) which class is created,
            # it just knows it gets a 'Transport' object.
            vehicle: Transport = factory.get_transport_v2(req)
            print(f"Request: {req.ljust(5)} | {vehicle.deliver()}")
        
        except ValueError as e:
            print(f"Request: {req.ljust(5)} | Error: {e}")

if __name__ == "__main__":
    main()

--- Logistics App Started ---

Request: road  | Delivering cargo by land in a box.
Request: sea   | Delivering cargo by sea in a container.
Request: air   | Delivering small parcel by air.
Request: space | Error: Unknown transport type: space


# More Pythonic Way

#### Why this is "More Pythonic"
- Functions over Classes: We don't need a `LogisticsFactory` class. A simple function `get_transport()` is sufficient. Python functions are first-class objects.
- `Protocols` over `ABC`s: We use Protocol (Duck Typing). The classes `Truck` and `Ship` do not need to inherit from `Transport`. As long as they have the deliver method, they work.
- Dictionary Registry: instead of a long `if/elif` or `match/case chain`, we use a dictionary to map strings to Classes. This is faster and cleaner.

### THE PROTOCOL (Interface)

We use a Protocol to define the shape of the object. </br>
runtime_checkable' allows us to use isinstance() if needed. </br>

In [1]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Transport(Protocol):
    def deliver(self) -> str: ...

### THE CONCRETE PRODUCTS

Notice: NO inheritance from 'Transport'. </br>
These are simple, standalone classes. </br>

In [2]:
class Truck:
    def deliver(self) -> str:
        return "Truck: Delivering by land in a box."

class Ship:
    def deliver(self) -> str:
        return "Ship: Delivering by sea in a container."

class Drone:
    def deliver(self) -> str:
        return "Drone: Delivering by air."

### THE PYTHONIC FACTORY (Function + Registry)

A dictionary mapping keys to the Class Types (not instances) </br>
Type[Transport] means "Any class that implements the Transport protocol" </br>

In [4]:
from typing import Type

TRANSPORT_REGISTRY: dict[str, Type[Transport]] = {
    "road": Truck,
    "sea":  Ship,
    "air":  Drone,
}

def get_transport(mode: str) -> Transport:
    """
    The Factory Function.
    Looks up the class in the registry and instantiates it.
    """
    try:
        # 1. Fetch the class (e.g., Truck)
        vehicle_class = TRANSPORT_REGISTRY[mode.lower()]
        
        # 2. Instantiate it (e.g., Truck())
        return vehicle_class()
        
    except KeyError:
        raise ValueError(f"Unknown transport mode: {mode}")

### CLIENT CODE

In [5]:
def main():
    print("--- Pythonic Factory App ---")

    # List of requested modes
    requests = ["road", "sea", "air", "space"]

    for mode in requests:
        try:
            # The client simply calls the function
            vehicle = get_transport(mode)
            print(f"Mode '{mode}': {vehicle.deliver()}")
        
        except ValueError as e:
            print(f"Mode '{mode}': Error -> {e}")

if __name__ == "__main__":
    main()

--- Pythonic Factory App ---
Mode 'road': Truck: Delivering by land in a box.
Mode 'sea': Ship: Delivering by sea in a container.
Mode 'air': Drone: Delivering by air.
Mode 'space': Error -> Unknown transport mode: space
